# 🎯 Visualización de Detecciones SED
Lee `predicciones_clean.csv` generado por `infer_clean.py` y proporciona:
- **Distribución de clases** detectadas
- **Timeline** de detecciones por micrófono y día
- **Reproducción de audios** agrupados por día

In [ ]:
# ── DEPENDENCIAS ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from pathlib import Path
from IPython.display import display, Audio, HTML
import ipywidgets as widgets
import librosa as lb
import soundfile as sf
import io, warnings
warnings.filterwarnings('ignore')

# ── PATHS ────────────────────────────────────────────────────────────────────
_ROOT      = Path().resolve()
if _ROOT.name == 'notebooks':   # si Jupyter arranca desde notebooks/
    _ROOT  = _ROOT.parent

CSV_PATH   = _ROOT / 'data' / 'processed' / 'predicciones_clean.csv'
CLEAN_DIR  = _ROOT / 'data' / 'clean'
OUT_DIR    = _ROOT / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

CLASSES = {
    0: 'Horn',
    1: 'Siren',
    2: 'Pets',
    3: 'Physiological',
    4: 'Speech',
    5: 'Ring Tone',
    6: 'Vibrating',
    7: 'Notifications',
    8: 'Cry',
}

# Paleta de colores por clase
COLORS = [
    '#e74c3c', '#e67e22', '#f1c40f', '#2ecc71',
    '#3498db', '#9b59b6', '#1abc9c', '#e91e63', '#795548'
]
CLASS_COLOR = {i: COLORS[i % len(COLORS)] for i in CLASSES}

plt.style.use('dark_background')
print('✅ Entorno listo')
print(f'   CSV  : {CSV_PATH}')
print(f'   WAVs : {CLEAN_DIR}')

## 1 · Carga y limpieza de datos

In [ ]:
df = pd.read_csv(CSV_PATH)
df['timestamp_onset']  = pd.to_datetime(df['timestamp_onset'], format='ISO8601')
df['timestamp_offset'] = pd.to_datetime(df['timestamp_offset'], format='ISO8601')
df['class_id']   = df['class_id'].astype(int)
df['class_name'] = df['class_id'].map(CLASSES)
df['duration']   = (df['timestamp_offset'] - df['timestamp_onset']).dt.total_seconds()
df['date']       = df['timestamp_onset'].dt.date
df['hour']       = df['timestamp_onset'].dt.hour

print(f'Total detecciones : {len(df):,}')
print(f'Micrófonos        : {sorted(df.mic_id.unique())}')
print(f'Días cubiertos    : {sorted(df.date.unique())}')
df.head()

## 2 · Distribución global de clases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#1a1a2e')

# --- Barplot: recuento por clase ---
counts = df['class_name'].value_counts().sort_values(ascending=True)
colors_bar = [CLASS_COLOR[k] for k in counts.index.map({v: k for k,v in CLASSES.items()})]
ax = axes[0]
ax.set_facecolor('#16213e')
bars = ax.barh(counts.index, counts.values, color=colors_bar, edgecolor='none', height=0.6)
ax.set_xlabel('Nº detecciones', color='white')
ax.set_title('Recuento por clase', color='white', fontsize=13, pad=10)
ax.tick_params(colors='white')
for bar, val in zip(bars, counts.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', color='white', fontsize=9)
ax.spines[:].set_color('#444')

# --- Pie: porcentaje ---
ax2 = axes[1]
ax2.set_facecolor('#16213e')
vals_all = df['class_name'].value_counts()
pie_colors = [CLASS_COLOR[k] for k in vals_all.index.map({v: k for k,v in CLASSES.items()})]
wedges, texts, autotexts = ax2.pie(
    vals_all.values, labels=vals_all.index,
    colors=pie_colors, autopct='%1.1f%%',
    startangle=140, textprops={'color': 'white', 'fontsize': 9},
    wedgeprops={'edgecolor': '#1a1a2e', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(8)
ax2.set_title('Distribución (%)', color='white', fontsize=13, pad=10)

plt.tight_layout()
plt.savefig(OUT_DIR / 'dist_clases.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 3 · Distribución por hora del día (heatmap)

In [ ]:
pivot = df.pivot_table(index='class_name', columns='hour',
                       values='confidence', aggfunc='count', fill_value=0)

fig, ax = plt.subplots(figsize=(18, 5))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

im = ax.imshow(pivot.values, aspect='auto', cmap='magma', interpolation='nearest')
plt.colorbar(im, ax=ax, label='Nº detecciones')

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{h:02d}h' for h in pivot.columns], color='white', fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, color='white')
ax.set_title('Detecciones por clase y hora del día', color='white', fontsize=14)
ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig(OUT_DIR / 'heatmap_horas.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 4 · Timeline por día y micrófono

In [ ]:
def plot_timeline_day(day, df_day, ax_row):
    """Dibuja el timeline de un día (una fila de subplots, un subplot por micrófono)."""
    mics = sorted(df_day['mic_id'].unique())
    for col_idx, mic in enumerate(mics):
        ax = ax_row[col_idx]
        ax.set_facecolor('#16213e')
        df_mic = df_day[df_day['mic_id'] == mic].sort_values('timestamp_onset')

        for _, row in df_mic.iterrows():
            ax.barh(
                y=row['class_id'],
                width=(row['timestamp_offset'] - row['timestamp_onset']).total_seconds() / 60,
                left=(row['timestamp_onset'] - pd.Timestamp(day)).total_seconds() / 60,
                color=CLASS_COLOR[row['class_id']],
                alpha=min(1.0, row['confidence'] + 0.3),
                height=0.6,
            )

        ax.set_yticks(list(CLASSES.keys()))
        ax.set_yticklabels(list(CLASSES.values()), fontsize=7, color='white')
        ax.set_xlabel('Minutos desde 00:00', color='white', fontsize=8)
        ax.set_title(f'{day} – M{mic}', color='white', fontsize=10)
        ax.tick_params(colors='white')
        ax.spines[:].set_color('#444')

    for col_idx in range(len(mics), len(ax_row)):
        ax_row[col_idx].set_visible(False)


days = sorted(df['date'].unique())
max_mics = df['mic_id'].nunique()

fig, axes = plt.subplots(
    nrows=len(days), ncols=max_mics,
    figsize=(max_mics * 10, len(days) * 3 + 1),
    squeeze=False
)
fig.patch.set_facecolor('#1a1a2e')

for row_idx, day in enumerate(days):
    df_day = df[df['date'] == day]
    plot_timeline_day(day, df_day, axes[row_idx])

legend_patches = [mpatches.Patch(color=CLASS_COLOR[k], label=v) for k, v in CLASSES.items()]
fig.legend(handles=legend_patches, loc='lower center', ncol=9,
           facecolor='#16213e', edgecolor='#444',
           labelcolor='white', fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Timeline de detecciones por día y micrófono',
             color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'timeline_detecciones.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 5 · Confianza media por clase

In [ ]:
conf_stats = (df.groupby('class_name')['confidence']
              .agg(['mean', 'std', 'count'])
              .sort_values('mean', ascending=False)
              .rename(columns={'mean': 'Media', 'std': 'Desv. típica', 'count': 'N'}))

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

bar_colors = [CLASS_COLOR[{v:k for k,v in CLASSES.items()}[name]] for name in conf_stats.index]
bars = ax.bar(conf_stats.index, conf_stats['Media'],
              color=bar_colors, edgecolor='none', width=0.5,
              yerr=conf_stats['Desv. típica'], capsize=4, error_kw={'color':'white','elinewidth':1})

ax.set_ylabel('Confianza media', color='white')
ax.set_title('Confianza media por clase', color='white', fontsize=13)
ax.tick_params(colors='white', axis='both')
ax.set_ylim(0, 1)
ax.spines[:].set_color('#444')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'confianza_clases.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

display(conf_stats.style.background_gradient(cmap='magma', subset=['Media']))

## 6 · 🔊 Reproducción de audios por día

Selecciona un día y un micrófono para escuchar todos los clips de ese día concatenados.
También puedes filtrar por clase de sonido.

In [ ]:
import re as _re

SR = 16_000
_FN_RE = _re.compile(
    r'^(\d{4})(\d{2})(\d{2})_(\d{2})_(\d{2})_(\d{2})_(\d{4})_M(\d+)\.wav$',
    _re.IGNORECASE,
)

def _parse_fn(fname):
    m = _FN_RE.match(fname)
    if not m:
        return None, None
    yr,mo,dy,hh,mm,ss,ms4,mic = m.groups()
    from datetime import datetime as dt
    start = dt(int(yr),int(mo),int(dy),int(hh),int(mm),int(ss),int(ms4)//10*1000)
    return int(mic), start

# Indexar todos los ficheros en clean/
index = []
for wav in sorted(CLEAN_DIR.glob('*.wav')):
    mic, start = _parse_fn(wav.name)
    if mic is not None:
        index.append({'path': wav, 'mic': mic, 'start': start,
                      'date': start.date()})

idx_df = pd.DataFrame(index)
print(f'Ficheros indexados: {len(idx_df)}')
idx_df.head()

In [ ]:
from datetime import date as _date

days_available = sorted(idx_df['date'].unique())
mics_available = sorted(idx_df['mic'].unique())
classes_all    = ['Todas'] + list(CLASSES.values())

w_day   = widgets.Dropdown(options=[str(d) for d in days_available], description='📅 Día:')
w_mic   = widgets.Dropdown(options=[str(m) for m in mics_available], description='🎙️ Mic:')
w_class = widgets.Dropdown(options=classes_all, description='🔊 Clase:')
w_btn   = widgets.Button(description='▶ Reproducir', button_style='success')
w_out   = widgets.Output()

def on_play(b):
    selected_day  = _date.fromisoformat(w_day.value)
    selected_mic  = int(w_mic.value)
    selected_cls  = w_class.value

    # Ficheros del día+mic
    mask = (idx_df['date'] == selected_day) & (idx_df['mic'] == selected_mic)
    files_day = idx_df[mask].sort_values('start')

    with w_out:
        w_out.clear_output()
        if files_day.empty:
            print(f'⚠️  Sin ficheros para {selected_day} – M{selected_mic}')
            return

        # Si hay filtro de clase, obtener intervalos de tiempo
        if selected_cls != 'Todas':
            cls_id = {v:k for k,v in CLASSES.items()}[selected_cls]
            mask_det = ((df['date'] == selected_day) &
                        (df['mic_id'] == selected_mic) &
                        (df['class_id'] == cls_id))
            det_day = df[mask_det]
            if det_day.empty:
                print(f'⚠️  Sin detecciones de "{selected_cls}" ese día/mic')
                return

        print(f'🔄  Cargando {len(files_day)} ficheros…')

        # Concatenar audio del día
        segments = []
        for _, row in files_day.iterrows():
            try:
                audio, _ = lb.load(str(row['path']), sr=SR, mono=True)
                segments.append((row['start'], audio))
            except Exception as e:
                print(f'  [SKIP] {row["path"].name}: {e}')

        if not segments:
            print('⚠️  No se pudo cargar ningún audio')
            return

        # Construir buffer continuo
        day_start = pd.Timestamp(selected_day)

        if selected_cls == 'Todas':
            # Reproducir todo el día concatenado
            full_audio = np.concatenate([s[1] for s in segments])
            print(f'▶  Audio completo · {len(full_audio)/SR:.1f} s')
            display(Audio(full_audio, rate=SR, normalize=True))
        else:
            # Extraer solo los segmentos detectados de esa clase
            # Construir buffer por offset en segundos
            clips = []
            full = np.concatenate([s[1] for s in segments])

            for _, det in det_day.iterrows():
                t0 = (det['timestamp_onset']  - day_start).total_seconds()
                t1 = (det['timestamp_offset'] - day_start).total_seconds()
                # Offset desde el primer fichero del día
                day_t0 = (segments[0][0] - day_start.to_pydatetime()).total_seconds()
                i0 = max(0, int((t0 - day_t0) * SR))
                i1 = min(len(full), int((t1 - day_t0) * SR))
                if i1 > i0:
                    clips.append(full[i0:i1])

            if not clips:
                print('⚠️  Los intervalos de detección no coinciden con los audios cargados')
                return

            clip_audio = np.concatenate(clips)
            print(f'▶  {len(clips)} clips de "{selected_cls}" · {len(clip_audio)/SR:.1f} s total')
            display(Audio(clip_audio, rate=SR, normalize=True))

w_btn.on_click(on_play)

ui = widgets.VBox([
    widgets.HTML('<h3 style="color:#3498db">🔊 Reproductor por día</h3>'),
    widgets.HBox([w_day, w_mic, w_class]),
    w_btn,
    w_out
])
display(ui)

## 7 · Resumen estadístico por día

In [ ]:
summary = (df.groupby(['date', 'class_name'])
           .agg(n_det=('confidence','count'),
                conf_media=('confidence','mean'),
                dur_total=('duration','sum'))
           .reset_index())

fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

pivot_day = summary.pivot_table(index='date', columns='class_name',
                                 values='n_det', fill_value=0)
bottom = np.zeros(len(pivot_day))
for cls_name in pivot_day.columns:
    cls_id = {v:k for k,v in CLASSES.items()}.get(cls_name, 0)
    ax.bar(pivot_day.index.astype(str), pivot_day[cls_name],
           bottom=bottom, label=cls_name,
           color=CLASS_COLOR[cls_id], edgecolor='none')
    bottom += pivot_day[cls_name].values

ax.set_xlabel('Día', color='white')
ax.set_ylabel('Nº detecciones', color='white')
ax.set_title('Detecciones por día (apiladas por clase)', color='white', fontsize=13)
ax.tick_params(colors='white', axis='both')
ax.spines[:].set_color('#444')
plt.xticks(rotation=30, ha='right')
ax.legend(loc='upper right', facecolor='#16213e', edgecolor='#444',
          labelcolor='white', fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'detecciones_por_dia.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

display(summary.pivot_table(index='date', columns='class_name',
                             values='n_det', fill_value=0)
               .style.background_gradient(cmap='Blues', axis=None))